# Caderno 01 — Modelos de Linguagem e Processamento de Linguagem Natural com HuggingFace

**Objetivo:** Demonstrar o uso de modelos pre-treinados do ecossistema HuggingFace
aplicados ao dominio de bulas medicas brasileiras.

**Rubrica 1:** Construir aplicacoes de Processamento de Linguagem Natural com
Modelos de Linguagem de Grande Escala e ecossistema HuggingFace (5 itens).

### Fluxo deste caderno
1. **AutoModel + AutoTokenizer** — carregar modelo, tokenizar, inspecionar dimensoes
2. **Analise de sentimento** — pipeline em frases clinicas, demonstrar limitacoes
3. **Reconhecimento de entidades nomeadas** — extrair medicamentos de bulas reais
4. **Tabela comparativa** — encoder-only vs decoder-only vs encoder-decoder
5. **Conclusao** — quais tarefas importam para o detector de interacoes


In [ ]:
import os
import sys
import logging
from pathlib import Path
from datetime import datetime

import torch
from transformers import pipeline, AutoModel, AutoTokenizer

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"

modelo_reconhecimento_entidades = "pucpr/clinicalnerpt-chemical"
modelo_embeddings = "neuralmind/bert-base-portuguese-cased"

diretorio_logs = Path("logs")
diretorio_logs.mkdir(exist_ok=True)

formato_log = logging.Formatter(
    fmt="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

arquivo_log = diretorio_logs / "caderno_01_modelos_linguagem.log"
manipulador_arquivo = logging.FileHandler(arquivo_log, encoding="utf-8")
manipulador_arquivo.setFormatter(formato_log)

manipulador_console = logging.StreamHandler(sys.stdout)
manipulador_console.setFormatter(formato_log)

registro = logging.getLogger("caderno_01")
registro.setLevel(logging.INFO)
registro.addHandler(manipulador_arquivo)
registro.addHandler(manipulador_console)

registro.info("=" * 70)
registro.info("Caderno 01 — Modelos de Linguagem e Processamento de Linguagem Natural")
registro.info("Inicio: %s", datetime.now().isoformat())
registro.info("PyTorch: %s", torch.__version__)
registro.info("CUDA disponivel: %s", torch.cuda.is_available())
registro.info("Dispositivo: %s", dispositivo)

if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    registro.info("GPU: %s | VRAM: %.1f GB",
                  torch.cuda.get_device_name(0), prop.total_memory / 1e9)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"Dispositivo: {dispositivo}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Logs: {arquivo_log.resolve()}")


## 2.1 Carregamento de Modelo com AutoModel e AutoTokenizer

Seguindo o padrao demonstrado em aula pelo professor:
1. `AutoTokenizer.from_pretrained(identificador)` — carrega o tokenizador
2. `AutoModel.from_pretrained(identificador)` — carrega o corpo do modelo
   **sem cabecalho de tarefa**
3. `modelo(**entradas)` — propagacao direta, retorna `last_hidden_state`

**Modelo:** `pucpr/clinicalnerpt-chemical` — BERT (apenas codificador)
treinado para reconhecimento de entidades em textos clinicos em portugues.

### Por que apenas codificador (encoder-only)?

- Atencao **bidirecional**: cada token enxerga toda a frase
- Ideal para tarefas de **compreensao**: classificacao, NER, embeddings
- Tokenizacao WordPiece: palavras raras quebradas em sub-tokens
  (exemplo: "poderoso" -> "poder", "##oso")


In [ ]:
identificador_modelo = modelo_reconhecimento_entidades
registro.info("Carregando tokenizador e modelo: %s", identificador_modelo)

tokenizador = AutoTokenizer.from_pretrained(identificador_modelo)
registro.info("Tokenizador: vocabulario com %d tokens", tokenizador.vocab_size)

modelo_codificador = AutoModel.from_pretrained(identificador_modelo).to(dispositivo)
registro.info("Modelo carregado em: %s", dispositivo)

frase_exemplo = "O mecanismo de atencao e poderoso"
entradas = tokenizador(frase_exemplo, return_tensors="pt")
entradas = {k: v.to(dispositivo) for k, v in entradas.items()}

with torch.no_grad():
    saidas = modelo_codificador(**entradas)

dims = saidas.last_hidden_state.shape
print(f"Dimensoes estado oculto: {dims}")
print(f"  Lote (batch):     {dims[0]}")
print(f"  Tokens:           {dims[1]} (inclui [CLS] e [SEP])")
print(f"  Dimensao oculta:  {dims[2]}")

tokens_gerados = tokenizador.convert_ids_to_tokens(entradas["input_ids"][0])
print(f"\nTokens WordPiece: {tokens_gerados}")
print(f"Total: {len(tokens_gerados)} (limite BERT: 512)")

registro.info("Dimensoes: [batch=%d, tokens=%d, dim=%d]",
              dims[0], dims[1], dims[2])
registro.info("Tokens: %s", tokens_gerados)


## 2.2 Analise de Sentimento em Frases Clinicas

Usamos o pipeline de analise de sentimento para classificar frases
extraidas de bulas medicas. O objetivo **nao e** obter resultados perfeitos,
mas **demonstrar as limitacoes** de um modelo generico (treinado em criticas
de filmes) quando aplicado a dominio especializado.

### Por que isso importa para o projeto?

Se usassemos analise de sentimento para classificar interacoes medicamentosas,
"aumenta a toxicidade" e "recomenda-se monitoramento" seriam ambos NEGATIVO —
mas o primeiro indica risco GRAVE e o segundo risco LEVE. A nuance se perde.
Isso **motiva** o uso de modelos especializados e ajuste fino.


In [ ]:
registro.info("Carregando pipeline de analise de sentimento...")

classificador_sentimento = pipeline("sentiment-analysis")

frases_clinicas = [
    "O uso concomitante e contraindicado devido ao risco de arritmia fatal.",
    "Nao ha interacoes conhecidas com este medicamento.",
    "Recomenda-se monitoramento da funcao renal durante o tratamento.",
    "A administracao de Amoxicilina com Metotrexato pode aumentar a toxicidade.",
    "O medicamento e seguro e bem tolerado pela maioria dos pacientes.",
]

print(f"{'Sentimento':>12} | {'Confianca':>9} | Frase")
print("-" * 75)

for frase in frases_clinicas:
    resultado = classificador_sentimento(frase)[0]
    sentimento = resultado["label"]
    confianca = resultado["score"]
    print(f"{sentimento:>12} | {confianca:>8.3f} | {frase[:55]}...")
    registro.info("Sentimento: %s (conf=%.3f) | %s",
                  sentimento, confianca, frase[:80])

registro.info("Analise de sentimento: %d frases", len(frases_clinicas))


## 2.3 Reconhecimento de Entidades Nomeadas com clinicalnerpt-chemical

**NER** classifica cada token como pertencente a uma entidade ou nao.

O modelo `pucpr/clinicalnerpt-chemical` identifica medicamentos em textos
clinicos em portugues, tanto principios ativos quanto nomes comerciais.

**Problema descoberto:** O modelo marca todos os sub-tokens como `B-ChemicalDrugs`
(sem `I-ChemicalDrugs`). Por isso, `aggregation_strategy="simple"` do pipeline
nao funciona corretamente. **Solucao:** agregacao manual por indice + prefixo `##`.

### Rotulos do modelo

- `B-ChemicalDrugs` — inicio de um nome de medicamento
- `I-ChemicalDrugs` — continuacao (nao usado por este modelo)
- `O` — fora de qualquer entidade


In [ ]:
def agregar_entidades(entidades):
    """Agrupa sub-tokens consecutivos em entidades completas.

    O modelo clinicalnerpt-chemical marca todos os sub-tokens como
    B-ChemicalDrugs (sem I-ChemicalDrugs). Agrupamos por indice
    consecutivo e prefixo ##.
    """
    if not entidades:
        return []

    grupos = []
    grupo_atual = [entidades[0]]

    for ent in entidades[1:]:
        anterior = grupo_atual[-1]
        if ent["index"] == anterior["index"] + 1 or ent["word"].startswith("##"):
            grupo_atual.append(ent)
        else:
            grupos.append(grupo_atual)
            grupo_atual = [ent]
    grupos.append(grupo_atual)

    resultados = []
    for grupo in grupos:
        palavra = "".join(t["word"].replace("##", "") for t in grupo)
        confianca = sum(t["score"] for t in grupo) / len(grupo)
        resultados.append({
            "word": palavra,
            "score": confianca,
            "start": grupo[0]["start"],
            "end": grupo[-1]["end"],
            "entity_group": grupo[0]["entity"],
        })
    return resultados

registro.info("Carregando pipeline de reconhecimento de entidades...")

reconhecedor_entidades = pipeline(
    "ner",
    model=modelo_reconhecimento_entidades,
    aggregation_strategy=None,
    device=0 if dispositivo == "cuda" else -1,
)
registro.info("Modelo NER: %s", modelo_reconhecimento_entidades)

trecho_bula_amoxicilina = (
    "A probenecida reduce a secrecao tubular renal da amoxicilina. "
    "No uso concomitante com amoxicilina, pode haver aumento dos niveis "
    "de amoxicilina no sangue. A administracao de alopurinol durante "
    "o tratamento com amoxicilina pode aumentar a probabilidade "
    "de reacoes alergicas. Existem casos raros de INR aumentada "
    "em pacientes mantidos com acenocumarol ou varfarina."
)

registro.info("Executando NER (%d caracteres)...", len(trecho_bula_amoxicilina))

entidades_raw = reconhecedor_entidades(trecho_bula_amoxicilina)
entidades_reconhecidas = agregar_entidades(entidades_raw)

print(f"{'Entidade':<22} {'Confianca':>9}  {'Inicio':>6}  {'Fim':>6}")
print("-" * 52)

for entidade in entidades_reconhecidas:
    print(f"{entidade['word']:<22} {entidade['score']:>8.3f}  "
          f"{entidade['start']:>6}  {entidade['end']:>6}")

medicamentos_unicos = sorted(set(e["word"] for e in entidades_reconhecidas))
print(f"\nMedicamentos unicos ({len(medicamentos_unicos)}):")
for m in medicamentos_unicos:
    print(f"  - {m}")

registro.info("NER: %d ocorrencias, %d unicos: %s",
              len(entidades_reconhecidas), len(medicamentos_unicos),
              medicamentos_unicos)


## 2.4 Tabela Comparativa de Arquiteturas

| Modelo | Arquitetura | Parametros | Tarefa | Limite | Dominio |
|--------|-------------|------------|--------|--------|---------|
| `clinicalnerpt-chemical` | Encoder-only (BERT) | 110M | NER | 512 | Clinico PT |
| DistilBERT (sentiment) | Encoder-only (BERT) | 66M | Sentimento | 512 | Geral EN |
| `biobertpt-all` | Encoder-only (BERT) | 110M | Classificacao | 512 | Biomedico PT |
| GPT-2 | Decoder-only | 124M | Geracao | 1024 | Geral |
| BART | Encoder-Decoder | 406M | Sumarizacao | 1024 | Geral EN |

### Encoder-only (BERT)
- Atencao bidirecional, treinamento com linguagem mascarada
- Ideal para: classificacao, NER, QA extrativa, embeddings

### Decoder-only (GPT-2)
- Atencao causal (so ve tokens anteriores), proximo token
- Ideal para: geracao de texto, chatbots

### Encoder-Decoder (BART)
- Codificador bidirecional + decodificador autoregressivo
- Ideal para: traducao, sumarizacao

| Abordagem | Vantagem | Desvantagem |
|-----------|---------|-------------|
| `pipeline()` | Uma linha, ja trata tokenizacao | Menos controle |
| `AutoModel` manual | Controle total, GPU explicita | Mais linhas |


## 2.5 Conclusao: Tarefas Importantes para o Detector

| Tarefa | Aplicacao | Onde |
|--------|-----------|------|
| **NER** | Extrair medicamentos da consulta e das bulas | Caderno 05 |
| **Classificacao** | Classificar risco: 0 (SEM), 1 (LEVE), 2 (GRAVE) | Cadernos 02, 05 |
| **Embeddings** | Busca vetorial em bulas | Cadernos 03, 05 |
| **Geracao (LLM)** | Resposta final fundamentada | Cadernos 02, 05 |
